# Decomposing a doubly stochastic matrix

## A Matrix Witness for Majorization

A square matrix $D$ is doubly stochastic when its entries are nonnegative and every row and column sums to one:

$$
D_{ij}\geq0,
\qquad
\sum_{j=1}^{n}D_{ij}=1,
\qquad
\sum_{i=1}^{n}D_{ij}=1.
$$

Here, $D_{ij}$ is the entry in row $i$ and column $j$, and $n$ is the matrix dimension. A target vector $y$ is majorized by a source vector $x$ exactly when there is a doubly stochastic matrix satisfying

$$
y=Dx.
$$

The matrix is a witness to the direction of the relation. It averages and redistributes the source components without changing their total.

In [1]:
from io import StringIO
from pathlib import Path
from zipfile import ZipFile
import hashlib
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI
from RiskLabAI.optimization import birkhoff_von_neumann_decomposition, is_doubly_stochastic, is_majorized, majorization_matrix

assert RiskLabAI.__version__ == "3.2.0"

DATA_PATH = ensure_dataset(BOOK_ROOT, "french_49_industries")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "24_portfolio_majorization"
TABLES = BOOK_ROOT / "outputs" / "tables" / "24_portfolio_majorization"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "24_portfolio_majorization"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)


def ordered(values):
    return np.sort(np.asarray(values, dtype=float))[::-1]


def partial_sums(values):
    return np.cumsum(ordered(values))


def concentration_record(name, values):
    weights = np.asarray(values, dtype=float)
    positive = weights[weights > 0]
    squared = float(np.sum(weights ** 2))
    entropy = float(-np.sum(positive * np.log(positive)))
    return {
        "Vector": name,
        "Largest component": float(np.max(weights)),
        "Squared sum": squared,
        "Fourth power sum": float(np.sum(weights ** 4)),
        "Effective count": float(1 / squared),
        "Entropy effective count": float(np.exp(entropy)),
    }


def load_public_returns():
    expected_hash = "8F394FE34BEA54D41B9AAFED410425EE8F8E252EDE3C71A7C1CD20BAB83040DE"
    assert DATA_PATH.is_file()
    assert sha256(DATA_PATH) == expected_hash
    with ZipFile(DATA_PATH) as archive:
        member = archive.namelist()[0]
        lines = archive.read(member).decode("latin1").splitlines()
    title_index = next(index for index, line in enumerate(lines) if "Average Value Weighted Returns -- Daily" in line)
    header_index = title_index + 1
    end_index = next(index for index in range(header_index + 1, len(lines)) if not lines[index].strip())
    frame = pd.read_csv(StringIO("\n".join(lines[header_index:end_index])))
    frame = frame.rename(columns={frame.columns[0]: "Date"})
    frame["Date"] = pd.to_datetime(frame["Date"].astype(str), format="%Y%m%d")
    industries = list(frame.columns[1:17])
    frame[industries] = frame[industries].apply(pd.to_numeric, errors="coerce")
    frame[industries] = frame[industries].mask(frame[industries] <= -99)
    frame = frame.loc[(frame["Date"] >= "2019-01-01") & (frame["Date"] <= "2025-12-31"), ["Date", *industries]].dropna().reset_index(drop=True)
    assert len(industries) == 16
    assert len(frame) == 1760
    return frame, industries

source = np.array([0.58, 0.24, 0.12, 0.06])
target = np.array([0.38, 0.28, 0.20, 0.14])
source_before = source.copy()
target_before = target.copy()
witness = majorization_matrix(target, source)
mapped = witness @ source

matrix_table = pd.DataFrame(witness, index=[f"Row {index + 1}" for index in range(4)], columns=[f"Column {index + 1}" for index in range(4)]).reset_index(names="Matrix row")
save_table(matrix_table, "stochastic_witness_matrix", "A doubly stochastic witness that maps the source to the target.", "tab:chapter25-stochastic-witness", fit=True)

sums = pd.DataFrame({
    "Index": np.arange(1, 5),
    "Row sum": witness.sum(axis=1),
    "Column sum": witness.sum(axis=0),
})
save_table(sums, "stochastic_row_column_sums", "Row and column sums of the witness matrix.", "tab:chapter25-stochastic-sums")

mapping = pd.DataFrame({
    "Component": np.arange(1, 5),
    "Source": source,
    "Target": target,
    "D times source": mapped,
    "Residual": mapped - target,
})
save_table(mapping, "witness_mapping", "Numerical check of the witness direction.", "tab:chapter25-witness-mapping")
mapping.to_csv(FIGURE_DATA / "witness_mapping.csv", index=False)

pd.DataFrame(witness).to_csv(FIGURE_DATA / "doubly_stochastic_matrix.csv", index=False)
figure, axis = plt.subplots(figsize=(7.0, 5.8), constrained_layout=True)
image = axis.imshow(witness, cmap="viridis", vmin=0, vmax=max(0.6, witness.max()))
axis.set_xticks(np.arange(4), labels=np.arange(1, 5))
axis.set_yticks(np.arange(4), labels=np.arange(1, 5))
axis.set_xlabel("Source component")
axis.set_ylabel("Target component")
axis.set_title("Doubly Stochastic Majorization Witness")
figure.colorbar(image, ax=axis, label="Matrix entry")
save_figure(figure, "doubly_stochastic_matrix")
plt.close(figure)

figure, axis = plt.subplots(figsize=(9.4, 5.2), constrained_layout=True)
positions = np.arange(4)
axis.bar(positions - 0.18, source, width=0.36, color=PALETTE["rose"], label="Source")
axis.bar(positions + 0.18, target, width=0.36, color=PALETTE["green"], label="Target")
axis.set_xticks(positions, labels=np.arange(1, 5))
axis.set_xlabel("Component")
axis.set_ylabel("Weight")
axis.set_title("Doubly Stochastic Mixing Reduces Concentration")
axis.legend()
finish_axis(axis)
save_figure(figure, "witness_mapping")
plt.close(figure)

,Matrix row,Column 1,Column 2,Column 3,Column 4
0,Row 1,0.565217,0,0.434783,0
1,Row 2,0.117647,0.882353,0,0
2,Row 3,0.163289,0.117647,0.565217,0.153846
3,Row 4,0.153846,0,0,0.846154


,Index,Row sum,Column sum
0,1,1,1
1,2,1,1
2,3,1,1
3,4,1,1


,Component,Source,Target,D times source,Residual
0,1,0.58,0.38,0.38,-1.66533e-16
1,2,0.24,0.28,0.28,0
2,3,0.12,0.2,0.2,0
3,4,0.06,0.14,0.14,5.55112e-17


<Figure size 700x580 with 2 Axes>

<Figure size 940x520 with 1 Axes>

## A Mixture of Permutations

Every $n\times n$ doubly stochastic matrix can be written as a convex mixture of permutation matrices:

$$
D=\sum_{r=1}^{R}\lambda_r P_r,
\qquad
\lambda_r>0,
\qquad
\sum_{r=1}^{R}\lambda_r=1.
$$

Here, $P_r$ is permutation matrix $r$, $\lambda_r$ is its mixture weight, and $R$ is the number of retained terms. The representation need not be unique. The numerical check must therefore verify reconstruction, positivity, and total weight rather than demand a particular list of permutations.

In [2]:
orders_given = [np.array([0, 1, 2, 3]), np.array([1, 2, 3, 0]), np.array([3, 2, 1, 0])]
weights_given = np.array([0.40, 0.35, 0.25])
mixture = sum(weight * np.eye(4)[order] for weight, order in zip(weights_given, orders_given))
mixture_before = mixture.copy()
weights_found, orders_found = birkhoff_von_neumann_decomposition(mixture)
restored = sum(weight * np.eye(4)[order] for weight, order in zip(weights_found, orders_found))

mixture_matrix = pd.DataFrame(mixture, index=[f"Row {index + 1}" for index in range(4)], columns=[f"Column {index + 1}" for index in range(4)]).reset_index(names="Matrix row")
save_table(mixture_matrix, "birkhoff_input_matrix", "Doubly stochastic matrix supplied to the decomposition.", "tab:chapter25-birkhoff-input", fit=True)

terms = pd.DataFrame([
    {
        "Term": index + 1,
        "Weight": weight,
        "Selected columns": " ".join(str(value + 1) for value in order),
    }
    for index, (weight, order) in enumerate(zip(weights_found, orders_found))
])
save_table(terms, "birkhoff_terms", "Checked Birkhoff decomposition returned by RiskLabAI.", "tab:chapter25-birkhoff-terms")
terms.to_csv(FIGURE_DATA / "birkhoff_weights.csv", index=False)

residual = restored - mixture
reconstruction = pd.DataFrame([
    {"Check": "Mixture weights sum to one", "Value": weights_found.sum(), "Pass": np.isclose(weights_found.sum(), 1, atol=1e-10, rtol=0)},
    {"Check": "All mixture weights are positive", "Value": weights_found.min(), "Pass": np.all(weights_found > 0)},
    {"Check": "Largest reconstruction residual", "Value": np.max(np.abs(residual)), "Pass": np.max(np.abs(residual)) <= 1e-10},
    {"Check": "Input matrix remains unchanged", "Value": np.max(np.abs(mixture - mixture_before)), "Pass": np.array_equal(mixture, mixture_before)},
])
save_table(reconstruction, "birkhoff_reconstruction_checks", "Numerical checks for the Birkhoff decomposition.", "tab:chapter25-birkhoff-reconstruction", fit=True)
pd.DataFrame(residual).to_csv(FIGURE_DATA / "birkhoff_reconstruction.csv", index=False)

figure, axis = plt.subplots(figsize=(8.7, 5.0), constrained_layout=True)
axis.bar(terms["Term"].astype(str), terms["Weight"], color=PALETTE["cyan"])
axis.set_xlabel("Returned permutation term")
axis.set_ylabel("Mixture weight")
axis.set_title("Positive Weights in the Returned Decomposition")
finish_axis(axis)
save_figure(figure, "birkhoff_weights")
plt.close(figure)

figure, axis = plt.subplots(figsize=(7.0, 5.6), constrained_layout=True)
limit = max(1e-16, np.max(np.abs(residual)))
image = axis.imshow(residual, cmap="coolwarm", vmin=-limit, vmax=limit)
axis.set_xticks(np.arange(4), labels=np.arange(1, 5))
axis.set_yticks(np.arange(4), labels=np.arange(1, 5))
axis.set_xlabel("Column")
axis.set_ylabel("Row")
axis.set_title("Reconstruction Residual")
figure.colorbar(image, ax=axis, label="Restored minus input")
save_figure(figure, "birkhoff_reconstruction")
plt.close(figure)

alphas = np.linspace(0, 1, 11)
path_rows = []
for alpha in alphas:
    path_matrix = (1 - alpha) * np.eye(4) + alpha * mixture
    mixed = path_matrix @ source
    path_rows.append({
        "Mixing fraction": alpha,
        "Squared concentration": np.sum(mixed ** 2),
        "Largest component": np.max(mixed),
        "Component standard deviation": np.std(mixed),
        "Target majorized by source": is_majorized(mixed, source),
    })
mixing_path = pd.DataFrame(path_rows)
save_table(mixing_path, "mixture_path_table", "Concentration along a convex path of doubly stochastic matrices.", "tab:chapter25-mixture-path", fit=True)
mixing_path.to_csv(FIGURE_DATA / "mixture_dispersion_path.csv", index=False)

figure, axis = plt.subplots(figsize=(9.4, 5.2), constrained_layout=True)
axis.plot(mixing_path["Mixing fraction"], mixing_path["Squared concentration"], marker="o", color=PALETTE["amber"], linewidth=2, label="Squared concentration")
axis.plot(mixing_path["Mixing fraction"], mixing_path["Largest component"], marker="s", color=PALETTE["rose"], linewidth=2, label="Largest component")
axis.set_xlabel("Mixing fraction")
axis.set_ylabel("Concentration summary")
axis.set_title("Every Mixed Vector Is No More Concentrated Than the Source")
axis.legend()
finish_axis(axis)
save_figure(figure, "mixture_dispersion_path")
plt.close(figure)

,Matrix row,Column 1,Column 2,Column 3,Column 4
0,Row 1,0.4,0.35,0,0.25
1,Row 2,0,0.4,0.6,0
2,Row 3,0,0.25,0.4,0.35
3,Row 4,0.6,0,0,0.4


,Term,Weight,Selected columns
0,1,0.4,1 2 3 4
1,2,0.35,2 3 4 1
2,3,0.25,4 3 2 1


,Check,Value,Pass
0,Mixture weights sum to one,1,Pass
1,All mixture weights are positive,0.25,Pass
2,Largest reconstruction residual,0,Pass
3,Input matrix remains unchanged,0,Pass


<Figure size 870x500 with 1 Axes>

<Figure size 700x560 with 2 Axes>

,Mixing fraction,Squared concentration,Largest component,Component standard deviation,Target majorized by source
0,0,0.412,0.58,0.201246,Pass
1,0.1,0.385266,0.5551,0.183893,Pass
2,0.2,0.361824,0.5302,0.167201,Pass
3,0.3,0.341675,0.5053,0.151389,Pass
4,0.4,0.324818,0.4804,0.136764,Pass
5,0.5,0.311252,0.4555,0.123746,Pass
6,0.6,0.30098,0.4306,0.112893,Pass
7,0.7,0.293999,0.4057,0.10488,Pass
8,0.8,0.29031,0.3808,0.100387,Pass
9,0.9,0.289914,0.3559,0.099893,Pass


<Figure size 940x520 with 1 Axes>

## Numerical Boundaries

The checks use an absolute tolerance $\varepsilon$. For a computed witness $\widehat D$, the largest mapping residual is

$$
r_{\max}=\max_i\left|\left(\widehat D x\right)_i-y_i\right|.
$$

Here, $x$ is the source, $y$ is the target, and $r_{\max}$ measures the largest componentwise numerical discrepancy. Passing a tolerance check confirms the stated finite calculation. It does not prove that a witness is unique or that a chosen financial allocation is desirable.

In [3]:
signed_source = np.array([-2.0, 0.0, 2.0])
signed_target = np.array([-1.0, 0.0, 1.0])
signed_witness = majorization_matrix(signed_target, signed_source)
signed_table = pd.DataFrame(signed_witness, index=[f"Row {index + 1}" for index in range(3)], columns=[f"Column {index + 1}" for index in range(3)]).reset_index(names="Matrix row")
save_table(signed_table, "signed_majorization_witness", "A checked witness for signed vectors with equal totals.", "tab:chapter25-signed-witness")

invalid_rows = []
for case, candidate in [
    ("Non-square matrix", np.array([[1.0, 0.0]])),
    ("Unequal column sums", np.array([[0.4, 0.6], [0.4, 0.6]])),
    ("Material negative entries", np.array([[1.1, -0.1], [-0.1, 1.1]])),
]:
    accepted = is_doubly_stochastic(candidate)
    try:
        birkhoff_von_neumann_decomposition(candidate)
        rejected = False
    except ValueError:
        rejected = True
    invalid_rows.append({"Case": case, "Accepted as doubly stochastic": accepted, "Decomposition rejects input": rejected})
invalid_table = pd.DataFrame(invalid_rows)
save_table(invalid_table, "invalid_matrix_checks", "Rejection checks for invalid matrix inputs.", "tab:chapter25-invalid-matrices", fit=True)

contract = pd.DataFrame([
    {"Check": "Target is majorized by source", "Pass": is_majorized(target, source), "Maximum residual": 0},
    {"Check": "Witness is doubly stochastic", "Pass": is_doubly_stochastic(witness), "Maximum residual": max(np.max(np.abs(witness.sum(axis=0) - 1)), np.max(np.abs(witness.sum(axis=1) - 1)))},
    {"Check": "Witness maps source to target", "Pass": np.allclose(mapped, target, atol=1e-10, rtol=0), "Maximum residual": np.max(np.abs(mapped - target))},
    {"Check": "Source input remains unchanged", "Pass": np.array_equal(source, source_before), "Maximum residual": np.max(np.abs(source - source_before))},
    {"Check": "Target input remains unchanged", "Pass": np.array_equal(target, target_before), "Maximum residual": np.max(np.abs(target - target_before))},
    {"Check": "Signed witness maps correctly", "Pass": np.allclose(signed_witness @ signed_source, signed_target, atol=1e-10, rtol=0), "Maximum residual": np.max(np.abs(signed_witness @ signed_source - signed_target))},
])
save_table(contract, "majorization_contract_checks", "Focused numerical checks of the released majorization contract.", "tab:chapter25-contract-checks", fit=True)

assert contract["Pass"].all()
assert invalid_table["Decomposition rejects input"].all()
assert np.all(mixing_path["Squared concentration"] <= mixing_path["Squared concentration"].iloc[0] + 1e-12)
assert np.all(mixing_path["Largest component"] <= mixing_path["Largest component"].iloc[0] + 1e-12)
display(contract)

,Matrix row,Column 1,Column 2,Column 3
0,Row 1,0.5,0.5,0
1,Row 2,0.5,0,0.5
2,Row 3,0,0.5,0.5


,Case,Accepted as doubly stochastic,Decomposition rejects input
0,Non-square matrix,Fail,Pass
1,Unequal column sums,Fail,Pass
2,Material negative entries,Fail,Pass


,Check,Pass,Maximum residual
0,Target is majorized by source,Pass,0
1,Witness is doubly stochastic,Pass,1.11022e-16
2,Witness maps source to target,Pass,1.66533e-16
3,Source input remains unchanged,Pass,0
4,Target input remains unchanged,Pass,0
5,Signed witness maps correctly,Pass,0


,Check,Pass,Maximum residual
0,Target is majorized by source,True,0.000000e+00
1,Witness is doubly stochastic,True,1.110223e-16
2,Witness maps source to target,True,1.665335e-16
3,Source input remains unchanged,True,0.000000e+00
4,Target input remains unchanged,True,0.000000e+00
5,Signed witness maps correctly,True,0.000000e+00
